<a href="https://colab.research.google.com/github/Dricajan/enem-2023-renda-desempenho/blob/main/Analise_exploratoria_enem_2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

O objetivo inicial dessa análise dos Microdados do ENEM 2023 é analisar a renda familiar dos participantes versus seu desempenho. O quanto a renda familiar afeta o desempenho? Além de explorar outros elementos que poderiam influenciar no desempenho final do participante do ENEM.

Dado o volume dos microdados do ENEM 2023, optou-se pelo processamento em ambiente cloud via Google Colab, prática alinhada com fluxos de trabalho modernos de ciência de dados para grandes volumes de dados.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Após análise do Dicionário de variáveis do pacote de microdados do ENEM, identificou-se colunas necessárias para a análise exploratória envolvendo renda versus desempenho no ENEM.

A população-alvo foi pensada a partir dos estudantes que estão concluindo o ensino médio no ano de 2023. Serão analisados o contexto socioeconômico do estudante que ainda frequenta o ambiente escolar, no qual há um contexto socioecônomico muito específico de localidade, tipo de escola e ensinamento síncrono do conteúdo abordado no ensino médio e consequentemente no ENEM. Essa escolha filtra participantes que já formaram há mais tempo, o que traz a possibilidade de já estarem inseridos no mercado de trabalho, além de poder estarem há mais tempo sem contato com o conteúdo formal de ensino, mudando o contexto social pretendido. Além disso, como nos microdados do ENEM não é especificado o local de moradia do estudante, partiu-se do presuposto que, ao estar frequentando uma escola atualmente, o estudante provavelmente reside nas proximidades, podendo considerar o contexto socioeconômico da região.

Outro filtro usado previamente foi de alunos não treineiros. Ou seja, que realmente pretendem fazer o ENEM no ano de 2023 com o objetivo de ingressar no curso superior.

Como o arquivo original dos microdados do ENEM possui aproximadamente 1.7Gb, optou-se pela análise dos dados usando o parâmetro chunksize para a melhor performance de processamento de grandes arquivos de dados.

Para essa primeira etapa da análise exploratória, criou-se uma tabela de mapeamento da distribuição da população-alvo ao longo do arquivo, para entender qual seria o melhor método de seleção da amostra que será usada na análise.

In [ ]:
db_monitoramento = pd.read_csv('/content/drive/MyDrive/Projeto_ENEM/MICRODADOS_ENEM_2023.csv', sep=';', chunksize=10000, encoding='latin-1', usecols=['TP_ST_CONCLUSAO','IN_TREINEIRO'] )
numero_pedaco = 0
dados = []
for pedaco in db_monitoramento:
   linha = []
   numero_pedaco +=1
   # Insere as posições de cada pedaço (chunk) na tabela
   linha.append(numero_pedaco)

   pedaco_filtrado = pedaco[(pedaco['TP_ST_CONCLUSAO']==2) & (pedaco['IN_TREINEIRO']==0)]
   resultado_contagem = len(pedaco_filtrado)
   # Insere apenas a quantidade de linhas da população alvo pretendida na tabela
   linha.append(resultado_contagem)
   dados.append(linha)

coluna = ['Número do pedaço','Quantidade da população-alvo']
# Cria dataframe final
df_mapeamento = pd.DataFrame(data=dados, columns=coluna)
# Retira o índice padrão da tabela
df_mapeamento.style.hide(axis='index')


In [ ]:
#Desenha o gráfico de linha baseado na tabela de mapeamento
df_mapeamento.plot(
    x='Número do pedaço',
    y='Quantidade da população-alvo',
    kind='line',
    figsize=(12, 5),
    title='Distribuição de Alunos do Ensino Médio ao Longo do Arquivo',
    grid=True
)

O laço for rodou por 394 vezes. Sendo que em cada um dos pedaços possui 10.000 linhas, há aproximadamente 3.940.000 linhas no arquivo de microdados do ENEM.
Dividindo 394 por 100%, em cada pedaço seria sorteado 4% da população-alvo em uma amostragem estratificada aleatória em cada chunk.

O parâmetro random_state foi colocado para garantir a mesma amostragem, mesmo em execuções diferentes do código.

In [ ]:
# Filtra somente as colunas pré-selecionadas para o objetivo de análise socioecônomica versus renda
db_filtro = pd.read_csv('/content/drive/MyDrive/Projeto_ENEM/MICRODADOS_ENEM_2023.csv', sep=';', chunksize=10000, encoding='latin-1', usecols=['TP_FAIXA_ETARIA','TP_COR_RACA','NO_MUNICIPIO_ESC',
'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_ST_CONCLUSAO','IN_TREINEIRO', 'Q001', 'Q002', 'Q005', 'Q006'])
tabela_filtro_01 = []
for pedaco in db_filtro:
  # Seleciona somente as linhas da população-alvo em cada pedaço
  pedaco_filtrado = pedaco[(pedaco['TP_ST_CONCLUSAO']==2) & (pedaco['IN_TREINEIRO']==0)]
  # Sorteia 4% de cada da quantidade da população-alvo em cada pedaço (chunk)
  amostra = pedaco_filtrado.sample(frac=0.04, random_state=42)
  # Elimina as colunas já filtradas da população-alvo
  amostra = amostra.drop(['TP_ST_CONCLUSAO', 'IN_TREINEIRO'], axis=1)
  tabela_filtro_01.append(amostra)
# Cria e concatena a tabela final
tabela_final = pd.concat(tabela_filtro_01)

In [ ]:
tabela_final.shape

In [ ]:
tabela_final.isnull().sum()

Verificou-se que nas colunas NU_NOTA, (referentes as notas nas diversas áreas do conhecimento), alguns registros possuem valores nulos. Os valores das notas são fundamentais, pois elas serão o principal fator de análise do desempenho dos estudantes.

In [ ]:
tabela_final.tail(100)

In [ ]:
db_monitoramento = pd.read_csv('/content/drive/MyDrive/Projeto_ENEM/MICRODADOS_ENEM_2023.csv', sep=';', chunksize=10000, encoding='latin-1', usecols=['TP_ST_CONCLUSAO','IN_TREINEIRO', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT'] )
numero_pedaco = 0
dados = []
for pedaco in db_monitoramento:
   linha = []
   numero_pedaco +=1
   linha.append(numero_pedaco)

   pedaco_filtrado = pedaco[(pedaco['TP_ST_CONCLUSAO']==2) & (pedaco['IN_TREINEIRO']==0) & (pedaco['TP_PRESENCA_CN']==1) & (pedaco['TP_PRESENCA_CH']==1) & (pedaco['TP_PRESENCA_LC']==1) & (pedaco['TP_PRESENCA_MT']==1)]
   resultado_contagem = len(pedaco_filtrado)

   linha.append(resultado_contagem)
   dados.append(linha)

coluna = ['Número do pedaço','Quantidade das linhas sob a condição']
df_mapeamento = pd.DataFrame(data=dados, columns=coluna)
df_mapeamento.style.hide(axis='index')


Diante da hipótese de que os valores nulos das notas estariam relacionados a ausência do participante na prova, inseriu-se como filtro as colunas relacionadas a presença nos dois dias de ENEM e nas provas de todas as áreas do conhecimento.

In [ ]:
# Adiciona as colunas de presença como filtro da população-alvo
db_filtro = pd.read_csv('/content/drive/MyDrive/Projeto_ENEM/MICRODADOS_ENEM_2023.csv', sep=';', chunksize=10000, encoding='latin-1', usecols=['TP_FAIXA_ETARIA','TP_COR_RACA','NO_MUNICIPIO_ESC',
'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_ST_CONCLUSAO','IN_TREINEIRO', 'Q001', 'Q002', 'Q005', 'Q006', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT'])
tabela_filtro_01 = []
for pedaco in db_filtro:
  # Filtro de estudantes concluindo o EM em 2023 + não treineiros + presentes nos 2 dias
  pedaco_filtrado = pedaco[(pedaco['TP_ST_CONCLUSAO']==2) & (pedaco['IN_TREINEIRO']==0) & (pedaco['TP_PRESENCA_CN']==1) & (pedaco['TP_PRESENCA_CH']==1) & (pedaco['TP_PRESENCA_LC']==1) & (pedaco['TP_PRESENCA_MT']==1)]
  # Mantendo o sorteio de 4% da pop-alvo em cada pedaço
  amostra = pedaco_filtrado.sample(frac=0.04, random_state=42)
  # Excluindo as colunas já usadas como filtro
  amostra = amostra.drop(['TP_ST_CONCLUSAO', 'IN_TREINEIRO', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT'], axis=1)
  tabela_filtro_01.append(amostra)
tabela_final = pd.concat(tabela_filtro_01)

In [ ]:
tabela_final.tail(100)

Verificando a quantidade de valores nulos e sua localização nas colunas

In [ ]:
tabela_final.isnull().sum()

Somente as colunas relacionadas a escola estão com valores nulos. Como dito antes, a localização das escolas é o principal indicador da região de residência do estudante, sendo fundamental sua presença. Logo, é necessário limpar os registros onde as informações sobre a escola são nulas.

O filtro das colunas de presença resolveu a presença de valores nulas nas colunas das notas.

In [ ]:
# Adiciona as colunas de presença como filtro da população-alvo
db_filtro = pd.read_csv('/content/drive/MyDrive/Projeto_ENEM/MICRODADOS_ENEM_2023.csv', sep=';', chunksize=10000, encoding='latin-1', usecols=['TP_FAIXA_ETARIA','TP_COR_RACA','NO_MUNICIPIO_ESC',
'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_ST_CONCLUSAO','IN_TREINEIRO', 'Q001', 'Q002', 'Q005', 'Q006', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT'])
tabela_filtro_01 = []
for pedaco in db_filtro:
  # Filtro de estudantes concluindo o EM em 2023 + não treineiros + presentes nos 2 dias + Municípios com valores não nulos
  pedaco_filtrado = pedaco[(pedaco['TP_ST_CONCLUSAO']==2) & (pedaco['IN_TREINEIRO']==0) & (pedaco['TP_PRESENCA_CN']==1) & (pedaco['TP_PRESENCA_CH']==1) & (pedaco['TP_PRESENCA_LC']==1) & (pedaco['TP_PRESENCA_MT']==1) & (pedaco['NO_MUNICIPIO_ESC'].notna())]
  # Mantendo o sorteio de 4% da pop-alvo em cada pedaço
  amostra = pedaco_filtrado.sample(frac=0.04, random_state=42)
  # Excluindo as colunas já usadas como filtro
  amostra = amostra.drop(['TP_ST_CONCLUSAO', 'IN_TREINEIRO', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT'], axis=1)
  tabela_filtro_01.append(amostra)
tabela_final = pd.concat(tabela_filtro_01)

In [ ]:
tabela_final.tail(100)

O filtro foi eficiente para eliminar todos os valores nulos de todas as colunas

In [ ]:
tabela_final.isnull().sum()

In [ ]:
tabela_final.shape

A amostra possui 28.853 indivíduos depois de todos os filtro inseridos. Para garantir a melhor representação da população na amostra, aumentou-se a porcentagem do sorteio, em cada pedaço (chunk), para 14%, na tentativa de garantir uma amostra com aproximadamente 100.000 indivíduos.

In [ ]:
# Adiciona as colunas de presença como filtro da população-alvo
db_filtro = pd.read_csv('/content/drive/MyDrive/Projeto_ENEM/MICRODADOS_ENEM_2023.csv', sep=';', chunksize=10000, encoding='latin-1', usecols=['TP_FAIXA_ETARIA','TP_COR_RACA','NO_MUNICIPIO_ESC',
'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_ST_CONCLUSAO','IN_TREINEIRO', 'Q001', 'Q002', 'Q005', 'Q006', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT'])
tabela_filtro_01 = []
for pedaco in db_filtro:
  # Filtro de estudantes concluindo o EM em 2023 + não treineiros + presentes nos 2 dias + Municípios com valores não nulos
  pedaco_filtrado = pedaco[(pedaco['TP_ST_CONCLUSAO']==2) & (pedaco['IN_TREINEIRO']==0) & (pedaco['TP_PRESENCA_CN']==1) & (pedaco['TP_PRESENCA_CH']==1) & (pedaco['TP_PRESENCA_LC']==1) & (pedaco['TP_PRESENCA_MT']==1) & (pedaco['NO_MUNICIPIO_ESC'].notna())]
  # Mantendo o sorteio de 14% da pop-alvo em cada pedaço
  amostra = pedaco_filtrado.sample(frac=0.14, random_state=42)
  # Excluindo as colunas já usadas como filtro
  amostra = amostra.drop(['TP_ST_CONCLUSAO', 'IN_TREINEIRO', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT'], axis=1)
  tabela_filtro_01.append(amostra)
tabela_final = pd.concat(tabela_filtro_01)

In [ ]:
tabela_final.shape

In [ ]:
tabela_final.isnull().sum()

Quantidade pretendida da amostra alcançada. Valores nulos continuam ausentes na nova amostragem.

Para facilitar a análise, fez-se um dicionário renomeando as colunas com nomes para facilitar o entendimento.

In [ ]:
# Dicionário de novos nomes das colunas
tabela_final_renomeada = tabela_final.rename(columns={
 'TP_FAIXA_ETARIA': 'faixa_etaria',
'TP_COR_RACA': 'cor_raca',
'NO_MUNICIPIO_ESC': 'municipio_escola',
'SG_UF_ESC': 'uf_escola',
'TP_DEPENDENCIA_ADM_ESC': 'tipo_da_escola',
'TP_LOCALIZACAO_ESC': 'local_da_escola',
'NU_NOTA_CN': 'nota_ciencias_da_natureza',
'NU_NOTA_CH': 'nota_ciencias_humanas',
'NU_NOTA_LC': 'nota_linguagens_e_codigo',
'NU_NOTA_MT': 'nota_matematica',
'NU_NOTA_REDACAO': 'nota_redacao',
'Q001': 'escolaridade_pai',
'Q002': 'escolaridade_mae',
'Q005': 'qta_moradores_residencia',
'Q006': 'renda_familiar'
})

In [ ]:
tabela_final_renomeada.head()

As colunas tipo_da_escola e local_da_escola estão como 'float'. Irei converter para 'int' para ser compatível com os numerais inteiros das categorias originais

In [ ]:
# Convertendo os valores da coluna tipo_da_escola para inteiros
tabela_final_renomeada['tipo_da_escola'] = tabela_final_renomeada['tipo_da_escola'].astype('Int64')

# Convertendo os valores da coluna local_da_escola para inteiros
tabela_final_renomeada['local_da_escola'] = tabela_final_renomeada['local_da_escola'].astype('Int64')

Como os microdados estão inseridos como categorias, criou-se um novo arquivo contendo e renomeando todas as categorias de todas as colunas para facilitar a análise exploratória.

No bloco abaixo, insiro no caminho do projeto a localização da pasta onde está o arquivo da análise exploratória e o arquico mapas.py criado com os dicionários de valores das colunas, o qual será importado para ser usado na tabela.

In [ ]:
import sys
caminho_projeto = '/content/drive/MyDrive/Projeto_ENEM'
# Evita duplicação ao rodar o código mais de uma vez
if caminho_projeto not in sys.path:
  sys.path.append(caminho_projeto)
# Importação do arquivo com todas as categorias renomeadas
import mapas

Cada coluna com valores a serem traduzidos foram tratadas pelos mapas criados. A condicional if foi usada para verificar se todos os valores da tabela estão contidos no mapa, caso verdadeiro ele é executado. Isso evita que o código seja rodado uma vez, traduzindo os valores, e em uma segunda rodagem, com os valores já alterados, estes não sejam encontrados no mapa resultando em valores nulos 'NaN'.

In [ ]:
if tabela_final_renomeada['faixa_etaria'].isin(mapas.mapa_faixa_etaria).all():
    tabela_final_renomeada['faixa_etaria'] = (
        tabela_final_renomeada['faixa_etaria'].map(mapas.mapa_faixa_etaria))

if tabela_final_renomeada['cor_raca'].isin(mapas.mapa_cor_raca).all():
  tabela_final_renomeada['cor_raca'] = (tabela_final_renomeada['cor_raca'].map(mapas.mapa_cor_raca))

if tabela_final_renomeada['tipo_da_escola'].isin(mapas.mapa_tipo_da_escola).all():
  tabela_final_renomeada['tipo_da_escola'] = tabela_final_renomeada['tipo_da_escola'].map(mapas.mapa_tipo_da_escola)

if tabela_final_renomeada['local_da_escola'].isin(mapas.mapa_localizacao_escola).all():
  tabela_final_renomeada['local_da_escola'] = tabela_final_renomeada['local_da_escola'].map(mapas.mapa_localizacao_escola)

if tabela_final_renomeada['escolaridade_pai'].isin(mapas.mapa_escolaridade_pai).all():
  tabela_final_renomeada['escolaridade_pai'] = tabela_final_renomeada['escolaridade_pai'].map(mapas.mapa_escolaridade_pai)

if tabela_final_renomeada['escolaridade_mae'].isin(mapas.mapa_escolaridade_mae).all():
  tabela_final_renomeada['escolaridade_mae'] = tabela_final_renomeada['escolaridade_mae'].map(mapas.mapa_escolaridade_mae)

if tabela_final_renomeada['renda_familiar'].isin(mapas.mapa_renda_familiar).all():
  tabela_final_renomeada['renda_familiar'] = tabela_final_renomeada['renda_familiar'].map(mapas.mapa_renda_familiar)

In [ ]:
tabela_final['Q006'].unique()

In [ ]:
tabela_final_renomeada.head()


In [ ]:
colunas_notas = tabela_final_renomeada[['nota_ciencias_da_natureza', 'nota_ciencias_humanas', 'nota_linguagens_e_codigo', 'nota_matematica', 'nota_redacao']]

tabela_final_renomeada['nota_geral'] = colunas_notas.mean(axis=1)
tabela_final_renomeada.head()

Será criada uma coluna de nota_geral. Esta representará uma medida sintética do desempenho do participante, obtida pela média simples das notas das cinco áreas avaliadas."

In [ ]:
tabela_final_renomeada['renda_familiar'] = pd.Categorical(
    tabela_final_renomeada['renda_familiar'],
    categories=mapas.ordem_renda,
    ordered=True
)

In [ ]:
df_renda_nota = tabela_final_renomeada.groupby('renda_familiar')['nota_geral'].mean().round(2).reset_index()
df_renda_nota

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df_renda_nota.plot(
     x='renda_familiar',
     y='nota_geral',
     kind = 'line',
     figsize=(15,5),
     title = 'Distribuição da Renda Familiar por Nota Geral',
     grid=True,
     rot = 45

)



Os valores em renda_familiar estão como texto (str) impossibilitando fazer cálculos. A coluna de renda com texto será mantida para fins de análise, mas será criada uma coluna com os pontos médios dos intervalos de faixa de renda, para fins de cálculo.

Os pontos médios serão criados como a média da amplitude do intervalo (menor valor + maior valor)/2.
'Nenhuma Renda' terá valor 0 (zero).

Para a primeira faixa aberta (até 1320 reais), foi considerado
o intervalo entre 0 (zero) e R$1.320 como aproximação, resultando em (0+1320)/2 = 660.

Para a faixa superior aberta (acima de 26.400,00 reais), foi adotado o valor de R$ 26.400 como representação numérica da categoria. Reconhece-se que essa escolha pode subestimar rendas superiores, uma vez que não há informação sobre o limite máximo da faixa. Optou-se por utilizar o limite inferior da categoria em vez de um valor arbitrário mais elevado, preservando uma abordagem conservadora para a transformação dos dados.

In [ ]:
if tabela_final_renomeada['renda_familiar'].isin(mapas.mapa_ponto_medio_renda_familiar).all():
  tabela_final_renomeada['ponto_medio_renda'] = (
        tabela_final_renomeada['renda_familiar'].map(mapas.mapa_ponto_medio_renda_familiar))
tabela_final_renomeada['ponto_medio_renda'] = tabela_final_renomeada['ponto_medio_renda'].astype(float)
tabela_final_renomeada.head(500)

In [ ]:
df_pt_renda_nota = tabela_final_renomeada.groupby('ponto_medio_renda')['nota_geral'].mean().reset_index()
df_pt_renda_nota.plot(
     x='ponto_medio_renda',
     y='nota_geral',
     kind = 'line',
     figsize=(15,5),
     title = 'Distribuição do Ponto médio das categorias de renda por Nota Geral',
     grid=True,
     rot = 45)

In [ ]:
tabela_final_renomeada['escolaridade_pai'] = pd.Categorical(
tabela_final_renomeada['escolaridade_pai'], categories=mapas.ordem_escolaridade_pais, ordered=True
)

In [ ]:
df_renda_escolaridade_pai = tabela_final_renomeada.groupby('escolaridade_pai')['nota_geral'].mean().round(2).reset_index()
df_renda_escolaridade_pai

In [ ]:
tabela_final_renomeada['escolaridade_mae'] = pd.Categorical(tabela_final_renomeada['escolaridade_mae'], categories=mapas.ordem_escolaridade_pais, ordered=True)

In [ ]:
df_renda_escolaridade_mae = tabela_final_renomeada.groupby('escolaridade_mae')['nota_geral'].mean().round(2).reset_index()
df_renda_escolaridade_mae

In [ ]:
df_renda_escolaridade_pai[df_renda_escolaridade_pai['escolaridade_pai'] != 'Não sei'].plot(
     x='escolaridade_pai',
     y='nota_geral',
     kind = 'line',
     figsize=(15,5),
     title = 'Distribuição da Escolaridade do Pai por Nota Geral',
     grid=True,
     rot = 45

)

In [ ]:
df_renda_escolaridade_mae[df_renda_escolaridade_mae['escolaridade_mae'] != 'Não sei'].plot(
     x='escolaridade_mae',
     y='nota_geral',
     kind = 'line',
     figsize=(15,5),
     title = 'Distribuição da Escolaridade da Mãe por Nota Geral',
     grid=True,
     rot = 45

)

In [ ]:
df_renda_escolaridade_pai_mae = pd.concat([df_renda_escolaridade_pai, df_renda_escolaridade_mae] , axis =1)
df_renda_escolaridade_pai_mae

Criação de tabela com as categorias de escolaridade e duas colunas: nota geral dos estudantes em relação a escolaridade do pai e nota geral dos estudantes em relação a escolaridade da mãe.

In [ ]:
# Renomeando 'nota_geral' em df_renda_escolaridade_pai para 'nota_geral_pai'
df_renda_escolaridade_pai_renomeada = df_renda_escolaridade_pai.rename(columns={'nota_geral': 'nota_geral_pai', 'escolaridade_pai': 'escolaridade'})

# Renomeando 'nota_geral' em df_renda_escolaridade_mae para 'nota_geral_mae'
df_renda_escolaridade_mae_renomeada = df_renda_escolaridade_mae.rename(columns={'nota_geral': 'nota_geral_mae', 'escolaridade_mae': 'escolaridade'})

# Merge os dois DataFrames a partir da coluna comum 'escolaridade'
df_escolaridade_combinado = pd.merge(df_renda_escolaridade_pai_renomeada, df_renda_escolaridade_mae_renomeada, on='escolaridade')

df_escolaridade_combinado

In [ ]:
# Filtra os DataFrames
df_pai_filtrado = df_renda_escolaridade_pai[df_renda_escolaridade_pai['escolaridade_pai'] != 'Não sei']
df_mae_filtrado = df_renda_escolaridade_mae[df_renda_escolaridade_mae['escolaridade_mae'] != 'Não sei']

# Plota o gráfico do Pai
eixo_x = df_pai_filtrado.plot(
    x='escolaridade_pai',
    y='nota_geral',
    kind='line',
    figsize=(15, 5),
    grid=True,
    label='Escolaridade do Pai'
)

# Plota o gráfico da Mãe no mesmo eixo
df_mae_filtrado.plot(
    x='escolaridade_mae',
    y='nota_geral',
    kind='line',
    ax=eixo_x,
    label='Escolaridade da Mãe'
)

# Ajustes de títulos e o giro do eixo X
plt.title('Distribuição da Escolaridade dos Pais por Nota Geral')
plt.xlabel('Escolaridade')
plt.ylabel('Nota Geral')

# Rotação das informações do eixo x
plt.xticks(rotation=45, ha='right')

plt.show()

Foram considerados apenas participantes que informaram a escolaridade de ambos os responsáveis, reduzindo a perda de informação e permitindo comparar famílias em que há registro das características educacionais dos dois membros parentais."


*   **Diferença salarial de gênero para a mesma escolaridade**: Mulheres ganhando menos do que homens com escolaridade semelhante. Isso poderia ser consequência da desigualdade de gênero no mercado de trabalho brasileiro.

*   **Estrututura familiar monoparental**: Como observado anteriormente. Um incremento da renda familiar correlaciona-se a um aumento de desemepnho na nota geral dos estudantes. Em uma família monoparental, há somente uma fonte de renda em relação a uma família com dois pais, o que poderia contribuir para uma renda familiar menor. Somando-se a isso, há o fato de a maioria das famílias monoparentais brasileiras serem formadas por mulheres. Sendo assim, famílias monoparentais formadas por mães poderia, pelo sua própria estrutura, contribuir para uma renda menor e, consequentemente influenciar em um menor desempenho dos estudantes dessas famílias.

Embora a informação sobre a escolaridade de ambos os responsáveis esteja disponível, os microdados não permitem identificar a composição familiar nem a participação financeira de cada responsável. Dessa forma, fatores como famílias monoparentais, guarda compartilhada ou diferentes níveis de contribuição financeira podem atuar como variáveis de confusão.





Para explorar melhor as possibilidades, irei comparar dados de tabelas do desempenho dos estudantes onde o pai tem no mínimo curso superior e a mãe no máximo ensino médio completo. E outra tabela onde a mãe tem no mínimo curso superior e o pai no máximo ensino médio completo.

In [ ]:
condicoes_pai_superior = tabela_final_renomeada['escolaridade_pai'].isin([
    'Completou a Faculdade, mas não completou a Pós-graduação',
    'Completou a Pós-graduação'
])

condicoes_mae_medio_ou_inferior = tabela_final_renomeada['escolaridade_mae'].isin([
    'Nunca estudou',
    'Não completou a 4ª série/5º ano do Ensino Fundamental',
    'Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino Fundamental',
    'Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio', 'Completou o Ensino Médio, mas não completou a Faculdade'
])

condicoes_mae_superior = tabela_final_renomeada['escolaridade_mae'].isin([    'Completou a Faculdade, mas não completou a Pós-graduação',
    'Completou a Pós-graduação'])

condicoes_pai_medio_ou_inferior = tabela_final_renomeada['escolaridade_pai'].isin(['Nunca estudou',
    'Não completou a 4ª série/5º ano do Ensino Fundamental',
    'Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino Fundamental',
    'Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio', 'Completou o Ensino Médio, mas não completou a Faculdade'])



df_superior_pai = tabela_final_renomeada[condicoes_pai_superior & condicoes_mae_medio_ou_inferior]
df_superior_mae = tabela_final_renomeada[condicoes_mae_superior & condicoes_pai_medio_ou_inferior]
df_superior_pai_mae = tabela_final_renomeada[condicoes_mae_superior & condicoes_pai_superior]
df_nenhum_superior_pai_mae = tabela_final_renomeada[condicoes_mae_medio_ou_inferior & condicoes_pai_medio_ou_inferior]

nota_pai_superior = df_superior_pai['nota_geral'].mean().round(2)
nota_mae_superior = df_superior_mae['nota_geral'].mean().round(2)
nota_mae_pai_superior = df_superior_pai_mae['nota_geral'].mean().round(2)
nota_mae_pai_nenhum_superior = df_nenhum_superior_pai_mae['nota_geral'].mean().round(2)

renda_pai_superior = df_superior_pai['ponto_medio_renda'].mean().round(2)
renda_mae_superior = df_superior_mae['ponto_medio_renda'].mean().round(2)
renda_mae_pai_superior = df_superior_pai_mae['ponto_medio_renda'].mean().round(2)
renda_mae_pai_nenhum_superior = df_nenhum_superior_pai_mae['ponto_medio_renda'].mean().round(2)


desempenho_pai_mae_superior  = {
   'Escolaridade': ['Só Mãe Superior', 'Só Pai Superior', 'Pai e Mãe Superior', 'Nem Pai, Nem Mãe Superior'],
   'Nota Geral': [nota_mae_superior, nota_pai_superior, nota_mae_pai_superior, nota_mae_pai_nenhum_superior],
   'Renda Média': [renda_mae_superior, renda_pai_superior, renda_mae_pai_superior, renda_mae_pai_nenhum_superior]

}
df_desempenho_pai_mae_superior = pd.DataFrame(desempenho_pai_mae_superior)
df_desempenho_pai_mae_superior


Quando o pai e a mae possuem no mínimo curso superior completo o desempenho dos alunos é maior que as outras alternativas.

Quando nenhum deles possui curso superior, os estudantes possuem o pior desempenho dentre as alternativas.

Quando somente um dos pais possui curso superior, e sendo este o pai, leva uma leve vantagem em relação a mãe como a única pessoa com superior, no desempenho do estudante.

A renda acompanha o desempenho dos estudantes. Alunos com melhores desempenhos vem de famílias com renda maior em relação as alternativas. Isso corrobora com a tabela geral da amostra que também relaciona maior renda com maior desempenho.

As famílias em que somente o pai possui ensino superior apresentaram renda média superior às famílias em que somente a mãe possui ensino superior. Essa diferença é acompanhada por uma pequena vantagem no desempenho médio dos estudantes.

Podemos fazer algumas observações e levantar hipóteses:

-> Uma possível explicação para a menor renda média observada nas famílias em que somente a mãe possui ensino superior é a desigualdade salarial entre homens e mulheres com escolaridade semelhante, fenômeno amplamente documentado na literatura brasileira. Entretanto, essa hipótese não pode ser testada diretamente com os microdados do ENEM.

-> Nesses casos assumimos que o membro parental com menor escolaridade tem no máximo ensino médio completo. Será que a escolaridade do pai no caso de somente a mãe ter superior não pode ser maior que no caso de somente o pai co msuperior e a mãe com no máximo ensino médio? Por exemplo:
   - Pai superior e mãe com ensino fundamental completo
   - Mãe superior e pai com ensino médio completo

Caso esta configuração seja mais frequente pode justificar uma renda maior por parte de somente o pai ter superior.


In [ ]:
condicoes_mae_nunca_estudou = tabela_final_renomeada['escolaridade_mae'].isin(['Nunca estudou'])
condicoes_mae_4_serie_fundamental = tabela_final_renomeada['escolaridade_mae'].isin(['Não completou a 4ª série/5º ano do Ensino Fundamental'])
condicoes_mae_8_serie_fundamental = tabela_final_renomeada['escolaridade_mae'].isin(['Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino Fundamental'])
condicoes_mae_ens_medio_inc = tabela_final_renomeada['escolaridade_mae'].isin(['Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio'])
condicoes_mae_ens_medio_comp = tabela_final_renomeada['escolaridade_mae'].isin(['Completou o Ensino Médio, mas não completou a Faculdade'])

df_superior_pai_mae_nunca_estudou = tabela_final_renomeada[condicoes_pai_superior & condicoes_mae_nunca_estudou]

df_superior_pai_mae_4_serie_fundamental = tabela_final_renomeada[condicoes_pai_superior & condicoes_mae_4_serie_fundamental]

df_superior_pai_mae_8_serie_fundamental = tabela_final_renomeada[condicoes_pai_superior & condicoes_mae_8_serie_fundamental]

df_superior_pai_mae_ens_medio_inc = tabela_final_renomeada[condicoes_pai_superior & condicoes_mae_ens_medio_inc]

df_superior_pai_mae_ens_medio_comp = tabela_final_renomeada[condicoes_pai_superior & condicoes_mae_ens_medio_comp]


qte_superior_pai_mae_nunca_estudou = df_superior_pai_mae_nunca_estudou['escolaridade_pai'].count()
qte_superior_pai_mae_4_serie_fundamental = df_superior_pai_mae_4_serie_fundamental['escolaridade_pai'].count()
qte_superior_pai_mae_8_serie_fundamental = df_superior_pai_mae_8_serie_fundamental['escolaridade_pai'].count()
qte_superior_pai_mae_ens_medio_inc = df_superior_pai_mae_ens_medio_inc['escolaridade_pai'].count()
qte_superior_pai_mae_ens_medio_comp = df_superior_pai_mae_ens_medio_comp['escolaridade_pai'].count()


escolaridade_mae_quando_pai_superior = {
   'Escolaridade Mãe': ['Nunca estudou', 'Não completou a 4ª série/5º ano do Ensino Fundamental', 'Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino Fundamental', 'Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio', 'Completou o Ensino Médio, mas não completou a Faculdade'],

   'Quantidade': [qte_superior_pai_mae_nunca_estudou, qte_superior_pai_mae_4_serie_fundamental, qte_superior_pai_mae_8_serie_fundamental, qte_superior_pai_mae_ens_medio_inc, qte_superior_pai_mae_ens_medio_comp]

}

df_escolaridade_mae_quando_pai_superior = pd.DataFrame(escolaridade_mae_quando_pai_superior)

df_escolaridade_mae_quando_pai_superior['Percentual (%)'] = (
    df_escolaridade_mae_quando_pai_superior['Quantidade']
    / df_escolaridade_mae_quando_pai_superior['Quantidade'].sum()
    * 100
).round(2)
df_escolaridade_mae_quando_pai_superior

In [ ]:
condicoes_pai_nunca_estudou = tabela_final_renomeada['escolaridade_pai'].isin(['Nunca estudou'])
condicoes_pai_4_serie_fundamental = tabela_final_renomeada['escolaridade_pai'].isin(['Não completou a 4ª série/5º ano do Ensino Fundamental'])
condicoes_pai_8_serie_fundamental = tabela_final_renomeada['escolaridade_pai'].isin(['Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino Fundamental'])
condicoes_pai_ens_medio_inc = tabela_final_renomeada['escolaridade_pai'].isin(['Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio'])
condicoes_pai_ens_medio_comp = tabela_final_renomeada['escolaridade_pai'].isin(['Completou o Ensino Médio, mas não completou a Faculdade'])

df_superior_mae_pai_nunca_estudou = tabela_final_renomeada[condicoes_mae_superior & condicoes_pai_nunca_estudou]

df_superior_mae_pai_4_serie_fundamental = tabela_final_renomeada[condicoes_mae_superior & condicoes_pai_4_serie_fundamental]

df_superior_mae_pai_8_serie_fundamental = tabela_final_renomeada[condicoes_mae_superior & condicoes_pai_8_serie_fundamental]

df_superior_mae_pai_ens_medio_inc = tabela_final_renomeada[condicoes_mae_superior & condicoes_pai_ens_medio_inc]

df_superior_mae_pai_ens_medio_comp = tabela_final_renomeada[condicoes_mae_superior & condicoes_pai_ens_medio_comp]


qte_superior_mae_pai_nunca_estudou = df_superior_mae_pai_nunca_estudou['escolaridade_mae'].count()
qte_superior_mae_pai_4_serie_fundamental = df_superior_mae_pai_4_serie_fundamental['escolaridade_mae'].count()
qte_superior_mae_pai_8_serie_fundamental = df_superior_mae_pai_8_serie_fundamental['escolaridade_mae'].count()
qte_superior_mae_pai_ens_medio_inc = df_superior_mae_pai_ens_medio_inc['escolaridade_mae'].count()
qte_superior_mae_pai_ens_medio_comp = df_superior_mae_pai_ens_medio_comp['escolaridade_mae'].count()

escolaridade_pai_quando_mae_superior = {
   'Escolaridade Pai': ['Nunca estudou', 'Não completou a 4ª série/5º ano do Ensino Fundamental', 'Completou a 4ª série/5º ano, mas não completou a 8ª série/9º ano do Ensino Fundamental', 'Completou a 8ª série/9º ano do Ensino Fundamental, mas não completou o Ensino Médio', 'Completou o Ensino Médio, mas não completou a Faculdade'],

   'Quantidade': [qte_superior_mae_pai_nunca_estudou, qte_superior_mae_pai_4_serie_fundamental, qte_superior_mae_pai_8_serie_fundamental, qte_superior_mae_pai_ens_medio_inc, qte_superior_mae_pai_ens_medio_comp]

}

df_escolaridade_pai_quando_mae_superior = pd.DataFrame(escolaridade_pai_quando_mae_superior)

df_escolaridade_pai_quando_mae_superior['Percentual (%)'] = (
    df_escolaridade_pai_quando_mae_superior['Quantidade']
    / df_escolaridade_pai_quando_mae_superior['Quantidade'].sum()
    * 100
).round(2)
df_escolaridade_pai_quando_mae_superior

In [ ]:
df_escolaridade_mae_quando_pai_superior

Analisando os dados vemos que, quando somente a mãe tem ensino superior, 67% dos pais completaram o ensino médio. E quando somente o pai tem ensino superior, 85,6% das mães tem ensino médio completo.

Uma possível explicação para a maior renda média observada nas famílias em que somente o pai possui ensino superior é que, nesses casos, o outro responsável (a mãe) apresenta, com maior frequência, Ensino Médio completo. Como maior escolaridade tende a estar associada a maior renda, essa composição familiar pode contribuir para a diferença observada entre os grupos.

Uma possível explicação adicional é a existência de diferenças de rendimento entre homens e mulheres no mercado de trabalho. Essas diferenças podem decorrer tanto de desigualdades salariais entre profissionais de mesma ocupação quanto da distribuição desigual entre profissões e cargos com diferentes níveis de remuneração.

Embora a diferença salarial entre homens e mulheres permaneça como uma hipótese plausível, os resultados sugerem que ela, isoladamente, não explica a diferença de renda observada entre os grupos. A distribuição da escolaridade do segundo responsável mostrou-se distinta entre as famílias analisadas, indicando que a renda familiar provavelmente é influenciada por múltiplos fatores. Assim, a diferença observada pode ser resultado da combinação entre a escolaridade de ambos os responsáveis, possíveis desigualdades no mercado de trabalho e outras variáveis não mensuradas pelos microdados do ENEM.